In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from vllm_client import VLLMClient

generator = VLLMClient()

In [ ]:
from transformers import AutoTokenizer
import json
from pathlib import Path

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

In [ ]:
prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are an artist"},
        {"role": "user", "content": "Why does blue and orange form complementary colors?"},
    ],
    tokenize=False,
    add_bos=True,
    add_generation_prompt=True,
)

In [ ]:
first = generator(prompt, sampling_params={"n":1, "min_tokens":250, "max_tokens":1000, "temperature":0.0, "repetition_penalty":1.1, "top_k":1})

In [ ]:
second = generator(prompt, sampling_params={"n":1, "min_tokens":250, "max_tokens":1000, "temperature":0.0, "repetition_penalty":1.1, "top_k":1})

In [ ]:
first == second

In [ ]:
from langchain_community.graphs import Neo4jGraph
from neo4j import GraphDatabase, Result
from typing import Dict, Any
import pandas as pd

In [ ]:
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "12345678"
NEO4J_DATABASE = "causedesc"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),database=NEO4J_DATABASE)

def db_query(cypher: str, params: Dict[str, Any] = {}):
    """Executes a Cypher statement and returns a DataFrame"""
    return driver.execute_query(
        cypher, parameters_=params, result_transformer_=Result.to_df
    )

In [ ]:
proto = pd.read_csv(f"./data/proto_cleaned.csv").drop(columns=["Unnamed: 0"])

In [ ]:
query = \
"""
CALL db.index.vector.queryNodes('vector', 25, $source)
YIELD node AS source, score AS sourceScore
WHERE source:__Entity__ AND sourceScore > 0.5
WITH source, sourceScore, count{(source)--()} AS sourceDegree
ORDER BY sourceScore * 0.4 + log10(sourceDegree + 1) * 0.6 DESC
MATCH (source)-[r]->(destination:__Entity__)
RETURN source.id, source.description, type(r) as relationshipType, r.description, destination.id, destination.description, sourceScore, sourceDegree
ORDER BY relationshipType, sourceDegree DESC
"""

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

embedding = HuggingFaceEmbeddings(
    model_name="pritamdeka/S-PubMedBert-MS-MARCO",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

In [ ]:
db_query(query, params={"source": embedding.embed_query("Anxiety")}).head(20)